# C1. Base world

Does the two-level world work at band 1 on varied noise? The track C reference.

See `docs/plans/experiments-plan.md`, entry C1. Results are read from `artifacts/results/C1/`. Nothing here trains or samples.

## Run

Run these from the repo root, in order. `uv run erosion-sim list` shows what is already trained.

**1. Data.** Skip any that already exist.

```sh
sbatch slurm/datagen.sbatch main_512
```

**2. Train.** One command per model; each resumes if stopped.

```sh
uv run erosion-sim train C1 coarse
uv run erosion-sim train C1 fine
```

**3. Evaluate.**

```sh
uv run erosion-sim evaluate C1
```

On the cluster, `uv run erosion-sim submit C1` does steps 2 and 3 as Slurm jobs.

Then run the cells below.

## How to read this

**Setup.** A world is a stack of levels, coarsest first. Each level predicts the erosion change on
overlapping windows, using its parent's output as context, and the windows are blended. The world is run on
whole test runs and scored on the interior, away from the run's fixed edges. The run's own edges affect
drainage everywhere, which the world cannot see, so expect an error floor of about 10–15% of the change
(`docs/notes/datagen-measurements.md`).

**Entries (table rows).**

- each world by name: the full pipeline.
- `level:<label>`: one level alone, given the true parent change as context, as in training. Separates a
  level's own error from errors passed down the pipeline.
- `analytical`: the stream power law solved exactly on the whole run.
- `analytical_chunks`, `model_chunks`, `simulator_chunks`: that method on separate closed chunks, stitched.
  Each chunk's edges act as false outlets, so these show the cost of naive chunking.
- `analytical_blended`: analytical on chunks at half-chunk stride, blended.

**Metrics.** The same as the erosion notebooks (`rmse`, `skill`, `iou`, and so on: skill 1 is perfect and 0
is no change; IoU 1 is perfect; ratios 1 are right; errors low), plus:

| Metric | What it is | Good |
|---|---|---|
| `chunk_edge_rmse` | RMSE within a few cells of an internal chunk boundary, where chunking errors appear | Low, and close to the entry's overall `rmse` |
| `seconds_per_query` | Time per world query | Low |

**Seams.** How much neighboring windows disagree where they overlap, before blending, as a share of the
change there. One number per phase; the last phase is the result. Low means the windows already agree; high
means the blend averages different answers.

**Breakdown by `dT`.** Skill per elapsed time. Longer times should be harder; a sharp drop shows where the
model stops being useful.

**Examples.** Per row, the start terrain, then the height change of the simulator and each entry. Red is
lowered (erosion), blue raised (deposition). Look for visible chunk or window boundaries.

**Training plot.** Left: train and validation loss; both should fall and level off. Validation rising while
train keeps falling is overfitting (the best-validation checkpoint is the one scored). Right: validation
skill, `1 - MSE / MSE(no change)`: 1 is perfect, 0 is no better than copying the input. For generative
models it is a one-step estimate, so read its trend, not its value.

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.experiments import registry
from erosion_sim.experiments.results import read_results
from erosion_sim.experiments.runner import model_folder
from erosion_sim.training.train import plot_history

NAME = "C1"
exp = registry.get(NAME)
results = read_results(NAME)
print(exp.question)
{"models": {label: m["folder"] for label, m in results.manifest["models"].items()},
 **{k: results.manifest[k] for k in ("code", "time", "slurm_job")}}

## Training

In [ ]:
for label in exp.models:
    folder = model_folder(NAME, label)[0]
    print(label, folder)
    plot_history(json.loads((folder / "history.json").read_text()))

## Worlds, levels alone, and baselines, on the runs' interiors

In [ ]:
plots.table(results, plots.TABLE + ('chunk_edge_rmse', 'seconds_per_query'))

In [ ]:
if "dT" in results.scalars["by"]:
    plots.by_table(results, "dT", "skill", kinds=("world", "baseline"))

## Seams

How much neighboring windows disagree before blending, per level and phase, as a share of the change.

In [ ]:
for world, levels in results.scalars["seams"].items():
    for level, phases in levels.items():
        print(world, level, [f"{p['relative']:.1%}" for p in phases])

## Examples

In [ ]:
plots.plot_examples(results, kinds=("world", "baseline"))

## Conclusion

_Written after reading the results above._